In [2]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import TruncatedSVD
from scipy.sparse import csr_matrix

In [3]:
movies = pd.read_csv("movies.csv")
ratings = pd.read_csv("ratings.csv")

In [4]:
movies = movies.dropna()
ratings = ratings.dropna()

In [5]:
tfidf = TfidfVectorizer(stop_words='english')

tfidf_matrix = tfidf.fit_transform(movies['genres'])


In [6]:
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

In [7]:
indices = pd.Series(
    movies.index,
    index=movies['title']
).drop_duplicates()


In [8]:
def content_recommendations(title, top_n=10):

    if title not in indices:
        return []

    idx = indices[title]

    sim_scores = list(enumerate(cosine_sim[idx]))

    sim_scores = sorted(
        sim_scores,
        key=lambda x: x[1],
        reverse=True
    )

    sim_scores = sim_scores[1:top_n + 1]

    movie_indices = [i[0] for i in sim_scores]

    return movies.iloc[movie_indices][
        ['movieId', 'title']
    ]


In [9]:
user_movie_matrix = ratings.pivot_table(
    index='userId',
    columns='movieId',
    values='rating'
).fillna(0)

In [10]:
matrix = csr_matrix(user_movie_matrix.values)

In [11]:
svd = TruncatedSVD(
    n_components=20,
    random_state=42
)
matrix_reduced = svd.fit_transform(matrix)

In [12]:
predicted_ratings = np.dot(
    matrix_reduced,
    svd.components_
)

predicted_ratings_df = pd.DataFrame(
    predicted_ratings,
    columns=user_movie_matrix.columns,
    index=user_movie_matrix.index
)

In [13]:
def hybrid_recommendations(user_id, title, top_n=10):

    content_movies = content_recommendations(
        title,
        top_n=20
    )

    hybrid_scores = []

    for _, row in content_movies.iterrows():

        movie_id = row['movieId']
        movie_title = row['title']


        if user_id in predicted_ratings_df.index \
           and movie_id in predicted_ratings_df.columns:

            collaborative_score = predicted_ratings_df.loc[
                user_id,
                movie_id
            ]

        else:
            collaborative_score = 0


        idx1 = indices[title]
        idx2 = indices[movie_title]

        content_score = cosine_sim[idx1][idx2]


        final_score = (
            0.5 * collaborative_score
        ) + (
            0.5 * content_score
        )

        hybrid_scores.append(
            (movie_title, final_score)
        )

    hybrid_scores = sorted(
        hybrid_scores,
        key=lambda x: x[1],
        reverse=True
    )

    return hybrid_scores[:top_n]

In [14]:
if __name__ == "__main__":

    user_id = 1

    movie_title = "Toy Story (1995)"

    recommendations = hybrid_recommendations(
        user_id,
        movie_title
    )

    print("\nRecommended Movies:\n")

    for movie, score in recommendations:

        print(
            f"{movie} => Score: {round(score, 3)}"
        )


Recommended Movies:

Toy Story 2 (1999) => Score: 1.393
Antz (1998) => Score: 1.163
Lord of the Rings, The (1978) => Score: 0.838
Monsters, Inc. (2001) => Score: 0.757
Black Cauldron, The (1985) => Score: 0.746
Emperor's New Groove, The (2000) => Score: 0.693
Land Before Time, The (1988) => Score: 0.662
Adventures of Rocky and Bullwinkle, The (2000) => Score: 0.642
We're Back! A Dinosaur's Story (1993) => Score: 0.54
Tale of Despereaux, The (2008) => Score: 0.52


In [15]:
import pandas as pd
import numpy as np
import streamlit as st
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import TruncatedSVD
from scipy.sparse import csr_matrix


st.set_page_config(page_title="Hybrid Movie Recommender", layout="wide")
st.title("🎞️ نظام ترشيح الأفلام")


@st.cache_resource
def load_data():
    movies = pd.read_csv("movies.csv")
    ratings = pd.read_csv("ratings.csv")
    movies['genres'] = movies['genres'].fillna('').str.replace('|', ' ')
    return movies, ratings

movies, ratings = load_data()


tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(movies['genres'])
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)


user_movie_matrix = ratings.pivot_table(index='userId', columns='movieId', values='rating').fillna(0)
svd = TruncatedSVD(n_components=12, random_state=42)
matrix_reduced = svd.fit_transform(csr_matrix(user_movie_matrix.values))
predicted_ratings = np.dot(matrix_reduced, svd.components_)
predicted_df = pd.DataFrame(predicted_ratings, columns=user_movie_matrix.columns, index=user_movie_matrix.index)

def get_hybrid_rec(user_id, title, top_n=5):
    if title not in movies['title'].values: return []
    idx = movies[movies['title'] == title].index[0]
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)[1:21]

    results = []
    for i, score in sim_scores:
        m_id = movies.iloc[i]['movieId']
        m_title = movies.iloc[i]['title']

        collab_score = predicted_df.loc[user_id, m_id] if user_id in predicted_df.index and m_id in predicted_df.columns else 0
        final_score = (0.5 * collab_score) + (0.5 * score)
        results.append((m_title, movies.iloc[i]['genres'], final_score))

    return sorted(results, key=lambda x: x[2], reverse=True)[:top_n]


col1, col2 = st.columns(2)
with col1:
    u_id = st.number_input("أدخل معرف المستخدم (User ID):", min_value=1, value=1)
with col2:
    m_name = st.selectbox("اختر فيلماً أعجبك:", movies['title'].values)

if st.button("عرض الترشيحات"):
    recs = get_hybrid_rec(u_id, m_name)
    for t, g, s in recs:
        st.success(f"**{t}**")
        st.caption(f"التصنيف: {g} | درجة التوافق: {s:.2f}")

2026-05-12 14:15:49.705 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-12 14:15:49.706 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-12 14:15:50.043 
  command:

    streamlit run C:\Users\Lenovo\AppData\Local\Programs\Python\Python313\Lib\site-packages\ipykernel_launcher.py [ARGUMENTS]
2026-05-12 14:15:50.044 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-12 14:15:50.044 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-12 14:15:50.048 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-12 14:15:50.924 Thread 'MainThread': missing ScriptRunContext! This w